# Sprint 3 — Atenção passo a passo: do laço à multiplicação de matrizes

**O que a Sprint 2 entregou:** tensores `(B, T, emb_dim)`. Cada token virou um vetor
treinável e ganhou a informação de posição. Mas cada vetor ainda descreve um token
*isolado* — nada nele depende dos tokens ao redor. Numa frase como "The bank of the river",
o vetor de "bank" é exatamente o mesmo que apareceria em "The bank of the city".

**O que falta:** deixar cada token montar uma representação que *leve em conta* os outros.
É isso que a atenção faz, e a conta é mais simples do que a notação sugere: para cada token,
uma média ponderada de todos os tokens da sequência, onde o peso mede o quanto um importa
para o outro.

Este notebook faz essa conta três vezes sobre o mesmo exemplo — com laço para um token, com
laço para todos, e de uma vez com multiplicação de matrizes — mostrando que as três dão o
mesmo resultado. A forma matricial é a que o projeto usa daqui em diante; o laço existe
para que ela não seja uma caixa-preta.

**Nesta etapa não há nenhum peso treinável.** As próprias entradas fazem os papéis de
query, key e value. As matrizes `W_q`, `W_k` e `W_v` entram no próximo passo, em
`src/attention/self_attention.py`.

In [1]:
import sys
from pathlib import Path

# Convenção do projeto: `jupyter notebook` é aberto na raiz do repositório (ver README),
# então o kernel deste notebook roda com cwd = notebooks/sprint03/. Dois níveis acima é a raiz.
PROJECT_ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

import torch

from src.attention import EXAMPLE_INPUTS, EXAMPLE_WORDS, simplified_attention

inputs = EXAMPLE_INPUTS

print(f"Frase   : {' '.join(EXAMPLE_WORDS)}")
print(f"Shape   : {tuple(inputs.shape)}  (T={inputs.shape[0]} tokens, emb_dim={inputs.shape[1]})")
print()
for palavra, vetor in zip(EXAMPLE_WORDS, inputs):
    print(f"  {palavra:<8} " + "".join(f"{v:>8.2f}" for v in vetor))

Frase   : Your journey starts with one step
Shape   : (6, 3)  (T=6 tokens, emb_dim=3)

  Your         0.43    0.15    0.89
  journey      0.55    0.87    0.66
  starts       0.57    0.85    0.64
  with         0.22    0.58    0.33
  one          0.77    0.25    0.10
  step         0.05    0.80    0.55


## Passo 1 — O vetor de contexto de um token, com laço

Tomamos **"journey"** (posição 1) como *query*: o token que está perguntando "quem aqui
importa para mim?". A resposta sai em três etapas.

**1. Scores.** O produto escalar entre a query e cada token da frase, inclusive ela mesma.
O produto escalar é grande quando dois vetores apontam para direções parecidas, então ele
serve de medida crua de semelhança.

**2. Pesos.** Os scores passam por um softmax, que os transforma numa distribuição: todos
positivos, somando 1. Isso é o que permite ler o resultado como "quanto da atenção de
'journey' vai para cada token".

**3. Contexto.** A soma dos vetores de todos os tokens, cada um multiplicado pelo seu peso.

In [2]:
query = inputs[1]  # "journey"

# 1. Scores: um produto escalar por token da frase.
scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    scores_2[i] = torch.dot(x_i, query)

# 2. Pesos: softmax transforma os scores numa distribuição.
weights_2 = torch.softmax(scores_2, dim=0)

# 3. Contexto: soma ponderada, acumulada token a token.
z_2 = torch.zeros(inputs.shape[1])
for i, x_i in enumerate(inputs):
    z_2 += weights_2[i] * x_i

print(f"query = {EXAMPLE_WORDS[1]!r}\n")
print(f"{'token':<10}{'score':>9}{'peso':>9}")
for palavra, score, peso in zip(EXAMPLE_WORDS, scores_2, weights_2):
    print(f"{palavra:<10}{score:>9.4f}{peso:>9.4f}")

print(f"\nsoma dos pesos : {weights_2.sum():.4f}")
print(f"z2             : {z_2}")

query = 'journey'

token         score     peso
Your         0.9544   0.1385
journey      1.4950   0.2379
starts       1.4754   0.2333
with         0.8434   0.1240
one          0.7070   0.1082
step         1.0865   0.1581

soma dos pesos : 1.0000
z2             : tensor([0.4419, 0.6515, 0.5683])


O `z2` acima é o **checkpoint** deste passo: `tensor([0.4419, 0.6515, 0.5683])`, o mesmo
valor do cálculo feito à mão antes de existir código.

Vale olhar os pesos antes de seguir. "journey" distribui atenção de forma bastante uniforme
— nada aqui chega perto de "olhar só para um token". O maior peso é o dela mesma (0,2379),
seguido de perto por "starts" (0,2333), que é quase o mesmo vetor. E é isso que o produto
escalar mede: semelhança, e nada além dela. Não há conceito de sintaxe, de ordem ou sequer
de "eu mesmo" — só vetores mais ou menos parecidos.

## Passo 2 — Todos os tokens, ainda com laço

O passo 1 respondeu por um token. Repetir aquilo seis vezes, uma por query, preenche uma
matriz `(6, 6)`: a linha `i` são os pesos do token `i` sobre todos os outros.

In [3]:
T = inputs.shape[0]

scores_laco = torch.empty(T, T)
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        scores_laco[i, j] = torch.dot(x_i, x_j)

# dim=-1: cada LINHA vira uma distribuição. Normalizar na dimensão errada é o
# erro clássico aqui — daria "quanto cada token é olhado", não "para onde ele olha".
weights_laco = torch.softmax(scores_laco, dim=-1)

context_laco = torch.empty_like(inputs)
for i in range(T):
    acumulado = torch.zeros(inputs.shape[1])
    for j, x_j in enumerate(inputs):
        acumulado += weights_laco[i, j] * x_j
    context_laco[i] = acumulado

cabecalho = "".join(f"{p:>9}" for p in EXAMPLE_WORDS)
print(f"pesos (linha = query)\n\n{'':<10}{cabecalho}")
for palavra, linha in zip(EXAMPLE_WORDS, weights_laco):
    print(f"{palavra:<10}" + "".join(f"{v:>9.4f}" for v in linha))

print(f"\nsoma de cada linha: {weights_laco.sum(dim=-1)}")

pesos (linha = query)

               Your  journey   starts     with      one     step
Your         0.2098   0.2006   0.1981   0.1242   0.1220   0.1452
journey      0.1385   0.2379   0.2333   0.1240   0.1082   0.1581
starts       0.1390   0.2369   0.2326   0.1242   0.1108   0.1565
with         0.1435   0.2074   0.2046   0.1462   0.1263   0.1720
one          0.1526   0.1958   0.1975   0.1367   0.1879   0.1295
step         0.1385   0.2184   0.2128   0.1420   0.0988   0.1896

soma de cada linha: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


Duas propriedades desta matriz merecem registro, porque as duas deixam de valer nos próximos
passos — e é justamente por isso que os próximos passos existem.

**Os scores são simétricos.** O produto escalar comuta, então o score entre "journey" e
"starts" é um número só, lido dos dois lados. Os *pesos* já não são simétricos (0,2369
contra 0,2333), porque o softmax normaliza linha a linha e cada linha tem sua própria soma.
Com `W_q` e `W_k` separados, nem os scores continuam simétricos — e isso é desejável: que
"the" dependa de "bank" não obriga "bank" a depender de "the" na mesma medida.

**Todo token vê a frase inteira**, inclusive o que vem depois dele. Para um modelo que é
treinado a prever o próximo token, isso é vazamento: a resposta está no material de
consulta. A máscara causal, em `src/attention/causal.py`, é o que fecha essa porta.

## Passo 3 — A mesma conta, com multiplicação de matrizes

O laço duplo do passo 2 calcula `dot(x_i, x_j)` para todo par `(i, j)`. Isso *é* a definição
de multiplicação de matrizes: `inputs @ inputs.T` produz exatamente a mesma tabela, com o
elemento `(i, j)` sendo o produto escalar da linha `i` pela linha `j`.

A forma matricial não é uma aproximação nem um algoritmo diferente — é o mesmo laço, escrito
de um jeito que a BLAS consegue paralelizar.

In [4]:
scores_matriz = inputs @ inputs.T
weights_matriz = torch.softmax(scores_matriz, dim=-1)
context_matriz = weights_matriz @ inputs

print(f"scores  idênticos aos do laço: {torch.allclose(scores_matriz, scores_laco)}")
print(f"pesos   idênticos aos do laço: {torch.allclose(weights_matriz, weights_laco)}")
print(f"contexto idêntico ao do laço : {torch.allclose(context_matriz, context_laco, atol=1e-6)}")
print(f"\nlinha de 'journey' bate com o z2 do passo 1: "
      f"{torch.allclose(context_matriz[1], z_2, atol=1e-6)}")

scores  idênticos aos do laço: True
pesos   idênticos aos do laço: True
contexto idêntico ao do laço : True

linha de 'journey' bate com o z2 do passo 1: True


## Passo 4 — Fechando o círculo com `src/attention/`

As três versões acima foram escritas dentro do notebook. A implementação que o resto do
projeto usa é `simplified_attention`, que chama a função núcleo
`scaled_dot_product_attention` com `Q = K = V = inputs` e `scale=False` — a atenção
simplificada é literalmente esse caso particular.

In [5]:
context_src, weights_src = simplified_attention(inputs)

print(f"pesos    conferem com o notebook: {torch.allclose(weights_src, weights_matriz)}")
print(f"contexto confere com o notebook : {torch.allclose(context_src, context_matriz, atol=1e-6)}")
print(f"\nz2 de src/attention/: {context_src[1]}")
print(f"z2 do passo 1       : {z_2}")
print(f"checkpoint          : {torch.allclose(context_src[1], torch.tensor([0.4419, 0.6515, 0.5683]), atol=1e-4)}")

pesos    conferem com o notebook: True
contexto confere com o notebook : True

z2 de src/attention/: tensor([0.4419, 0.6515, 0.5683])
z2 do passo 1       : tensor([0.4419, 0.6515, 0.5683])
checkpoint          : True


## O que falta

Esta versão já é atenção de verdade: cada token sai com um vetor que depende de todos os
outros. Mas ela não serve como camada de um modelo, por duas razões.

**Não tem o que aprender.** A saída é função apenas da entrada — nenhum parâmetro, nenhum
gradiente. O critério de "semelhança" está congelado no produto escalar cru dos embeddings,
e não há como o treino descobrir que, para prever a próxima palavra, certas relações
importam mais que a semelhança bruta de vetores. As matrizes `W_q`, `W_k` e `W_v` entram
exatamente aí: elas projetam os mesmos vetores em três espaços distintos, e é nesse espaço
aprendido que a comparação passa a acontecer. É o próximo passo,
`src/attention/self_attention.py`.

**Vê o futuro.** A linha de "journey" no passo 2 dá peso a "step", que vem quatro posições
adiante. Num modelo treinado para prever o próximo token, isso é ver a resposta. A máscara
causal — `True` na posição bloqueada, `-inf` antes do softmax, conforme a decisão registrada
em `src/attention/__init__.py` — resolve isso em `src/attention/causal.py`.